In [27]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [28]:
spark = SparkSession.builder \
    .appName("credit-risk-lakehouse-parcela") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

spark

In [29]:
BASE_PATH = "/app/data"

In [30]:
df_contrato = spark.read.parquet(
    f"{BASE_PATH}/bronze/contrato"
)

df_contrato.count()

18027

In [31]:
df_parcela = df_contrato \
    .withColumn(
        "numero_parcela",
        explode(
            sequence(
                lit(1),
                col("prazo_meses")
            )
        )
    )

df_parcela = df_parcela \
    .withColumn(
        "valor_parcela",
        round(
            col("valor_contratado") /
            col("prazo_meses"),
            2
        )
    )

df_parcela = df_parcela \
    .withColumn(
        "data_vencimento",
        add_months(
            col("data_contratacao"),
            col("numero_parcela")
        )
    )

df_parcela = df_parcela.withColumn(
    "status_parcela",
    when(col("data_vencimento") < current_date(),
         element_at(
             array(lit("paga"), lit("paga"), lit("paga"), lit("atrasada")),
             floor(rand() * 4 + 1).cast("int")
         )
    ).otherwise(lit("aberta"))
)

df_parcela = df_parcela \
    .withColumn(
        "parcela_id",
        monotonically_increasing_id()
    )

In [32]:
df_parcela = df_parcela.select(
    "parcela_id",
    "contrato_id",
    "numero_parcela",
    "valor_parcela",
    "data_vencimento",
    "status_parcela"
)

In [33]:
df_parcela.write \
    .mode("overwrite") \
    .parquet(f"{BASE_PATH}/bronze/parcela")

In [36]:
spark.stop()